In [ ]:
import os
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, ResNet152, ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from skimage import exposure
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import pandas as pd
from skimage.filters import gabor_kernel
from scipy import ndimage as ndi
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D
import pandas as pd
import os

In [ ]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalMaxPooling2D, GlobalAveragePooling2D, Concatenate, Input, Layer
from tensorflow.keras.models import Model

# --- ✅ EuPea Attention Module ---
class EuPea(tf.keras.layers.Layer):
    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):
        # Input: [B, H, W, C]
        x_mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)   # [B, 1, 1, C]
        x_std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)  # [B, 1, 1, C]

        # --- Euclidean Distance per-pixel per-channel ---
        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)  # [B, H, W, C]
        d_min = tf.reduce_min(d, axis=[1, 2], keepdims=True)
        d_max = tf.reduce_max(d, axis=[1, 2], keepdims=True)
        d_norm = (d - d_min) / (d_max - d_min + self.epsilon)  # [B, H, W, C]

        # --- Pearson Approximation (squared Z-score) ---
        z = tf.square((x - x_mean) / (x_std + self.epsilon))  # [B, H, W, C]
        z_min = tf.reduce_min(z, axis=[1, 2], keepdims=True)
        z_max = tf.reduce_max(z, axis=[1, 2], keepdims=True)
        z_norm = (z - z_min) / (z_max - z_min + self.epsilon)  # [B, H, W, C]

        # --- Combine Attention and Apply ---
        attention = tf.nn.sigmoid(d_norm + z_norm)  # [B, H, W, C]
        return x * attention

# --- ✅ Build ResNet50 Feature Extraction Model with EuPea ---
def build_eupea_resnet50_feature_model():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    input_tensor = Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet50.preprocess_input(input_tensor)

    backbone = Model(
        inputs=base_model.input,
        outputs=[
            base_model.get_layer("pool1_pool").output,         # 64 channels
            base_model.get_layer("conv2_block3_out").output,   # 256 channels
            base_model.get_layer("conv3_block4_out").output,   # 512 channels
            base_model.get_layer("conv4_block6_out").output,   # 1024 channels
            base_model.get_layer("conv5_block3_out").output    # 2048 channels
        ]
    )

    features = backbone(x)
    attention = EuPea()

    pooling_layers = [
        GlobalMaxPooling2D(),
        GlobalMaxPooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D()
    ]

    pooled_outputs = []
    for feat, pool in zip(features, pooling_layers):
        attn_feat = attention(feat)
        pooled_outputs.append(pool(attn_feat))

    final_feature = Concatenate()(pooled_outputs)
    return Model(inputs=input_tensor, outputs=final_feature)

# --- ✅ Instantiate Feature Model ---
eupea_feature_model = build_eupea_resnet50_feature_model()
print(eupea_feature_model.summary())

# --- ✅ Image Directory and Labels ---
# --- ✅ Define image directory and labels ---
image_dir = r"/kaggle/input/cervical-type/After Augmentation"
labels = ['Type - 1', 'Type - 2', 'Type - 3']
res_50_df = pd.DataFrame(columns=['features', 'labels'])

# --- ✅ Feature Extraction Function ---
def extract_features(image_path, label):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = tf.expand_dims(x, axis=0)

    features = eupea_feature_model(x)
    features_np = features.numpy().flatten()
    res_50_df.loc[len(res_50_df)] = [features_np, label]

# --- ✅ Process Dataset ---
for label_idx, label in enumerate(labels):
    label_dir = os.path.join(image_dir, label)
    for subdir, dirs, files in os.walk(label_dir):
        for file in files:
            extract_features(os.path.join(subdir, file), label_idx)

# --- ✅ Show Results ---
print(res_50_df.head())
print("Unique feature lengths:", res_50_df['features'].apply(len).unique())


In [ ]:
print("BF-AD-AP-Resnet50 dataframe", res_50_df)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, f1_score

# Separate features and labels
X = res_50_df['features'].tolist()
y = res_50_df['labels'].tolist()

print("ResNet50 with 5-fold cross-validation and linear SVM")

num_splits = 5

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

kfold = KFold(n_splits=num_splits, shuffle=True)
svm = SVC(kernel='linear')

fold = 1
for train_index, test_index in kfold.split(X):
    print(f"\n================ Fold {fold} ================")

    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    conf_matrix = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(conf_matrix)

    # ----- Per-class correct/incorrect -----
    classes = [0, 1, 2]
    print("\nPer-class prediction details:")
    for cls in classes:
        correct = conf_matrix[cls, cls]
        incorrect = conf_matrix[cls].sum() - correct
        total_samples = conf_matrix[cls].sum()

        print(f"  Class {cls}: Total = {total_samples}, "
              f"Correct = {correct}, Incorrect = {incorrect}")

    # Metrics calculation
    tp = conf_matrix[0, 0]
    fp = conf_matrix[1, 0] + conf_matrix[2, 0]
    fn = conf_matrix[0, 1] + conf_matrix[0, 2]
    tn = conf_matrix[1, 1] + conf_matrix[2, 2] + conf_matrix[1, 2] + conf_matrix[2, 1]

    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)

    fold += 1

# ---- Averages ----
average_accuracy = np.mean(accuracy_scores)
average_specificity = np.mean(specificity_scores)
average_sensitivity = np.mean(sensitivity_scores)
average_precision = np.mean(precision_scores)
average_f1 = np.mean(f1_scores)

print("\n============== Final Averages ==============")
print(f"Average Accuracy: {average_accuracy:.6f}")
print(f"Average Specificity: {average_specificity:.6f}")
print(f"Average Sensitivity: {average_sensitivity:.6f}")
print(f"Average Precision: {average_precision:.6f}")
print(f"Average F1 Score: {average_f1:.6f}")


In [ ]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.applications import ResNet101
from tensorflow.keras.layers import GlobalMaxPooling2D, GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.models import Model

# --- ✅ EuPea Attention Module ---
class EuPea(tf.keras.layers.Layer):
    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):
        x_mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        x_std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)

        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)
        d_min = tf.reduce_min(d, axis=[1, 2], keepdims=True)
        d_max = tf.reduce_max(d, axis=[1, 2], keepdims=True)
        d_norm = (d - d_min) / (d_max - d_min + self.epsilon)

        z = tf.square((x - x_mean) / (x_std + self.epsilon))
        z_min = tf.reduce_min(z, axis=[1, 2], keepdims=True)
        z_max = tf.reduce_max(z, axis=[1, 2], keepdims=True)
        z_norm = (z - z_min) / (z_max - z_min + self.epsilon)

        attention = tf.nn.sigmoid(d_norm + z_norm)
        return x * attention

# --- ✅ Build ResNet101 Feature Extraction Model with EuPea ---
def build_eupea_resnet101_feature_model():
    base_model = ResNet101(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    input_tensor = Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet.preprocess_input(input_tensor)

    backbone = Model(
        inputs=base_model.input,
        outputs=[
            base_model.get_layer("pool1_pool").output,          # 64 channels
            base_model.get_layer("conv2_block3_out").output,    # 256 channels
            base_model.get_layer("conv3_block4_out").output,    # 512 channels
            base_model.get_layer("conv4_block23_out").output,   # 1024 channels
            base_model.get_layer("conv5_block3_out").output     # 2048 channels
        ]
    )

    features = backbone(x)
    attention = EuPea()

    pooling_layers = [
        GlobalMaxPooling2D(),
        GlobalMaxPooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D()
    ]

    pooled_outputs = []
    for feat, pool in zip(features, pooling_layers):
        attn_feat = attention(feat)
        pooled_outputs.append(pool(attn_feat))

    final_feature = Concatenate()(pooled_outputs)
    return Model(inputs=input_tensor, outputs=final_feature)

# --- ✅ Instantiate Feature Model ---
eupea_feature_model = build_eupea_resnet101_feature_model()
print(eupea_feature_model.summary())

# --- ✅ Image Directory and Labels ---
# --- ✅ Define image directory and labels ---
image_dir = r"/kaggle/input/cervical-type/After Augmentation"
labels = ['Type - 1', 'Type - 2', 'Type - 3']
res_101_df = pd.DataFrame(columns=['features', 'labels'])

# --- ✅ Feature Extraction Function ---
def extract_features(image_path, label):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = tf.expand_dims(x, axis=0)

    features = eupea_feature_model(x)
    features_np = features.numpy().flatten()
    res_101_df.loc[len(res_101_df)] = [features_np, label]

# --- ✅ Process Dataset ---
for label_idx, label in enumerate(labels):
    label_dir = os.path.join(image_dir, label)
    for subdir, dirs, files in os.walk(label_dir):
        for file in files:
            extract_features(os.path.join(subdir, file), label_idx)

# --- ✅ Show Results ---
print(res_101_df.head())
print("Unique feature lengths:", res_101_df['features'].apply(len).unique())


In [ ]:
print("BF-AD-AP-Resnet101 dataframe", res_101_df)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, f1_score

# Separate features and labels
X = res_101_df['features'].tolist()
y = res_101_df['labels'].tolist()

print("ResNet50 with 5-fold cross-validation and linear SVM")

num_splits = 5

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

kfold = KFold(n_splits=num_splits, shuffle=True)
svm = SVC(kernel='linear')

fold = 1
for train_index, test_index in kfold.split(X):
    print(f"\n================ Fold {fold} ================")

    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    conf_matrix = confusion_matrix(y_test, y_pred)
    print("\nConfusion Matrix:")
    print(conf_matrix)

    # ----- Per-class correct/incorrect -----
    classes = [0, 1, 2]
    print("\nPer-class prediction details:")
    for cls in classes:
        correct = conf_matrix[cls, cls]
        incorrect = conf_matrix[cls].sum() - correct
        total_samples = conf_matrix[cls].sum()

        print(f"  Class {cls}: Total = {total_samples}, "
              f"Correct = {correct}, Incorrect = {incorrect}")

    # Metrics calculation
    tp = conf_matrix[0, 0]
    fp = conf_matrix[1, 0] + conf_matrix[2, 0]
    fn = conf_matrix[0, 1] + conf_matrix[0, 2]
    tn = conf_matrix[1, 1] + conf_matrix[2, 2] + conf_matrix[1, 2] + conf_matrix[2, 1]

    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)

    fold += 1

# ---- Averages ----
average_accuracy = np.mean(accuracy_scores)
average_specificity = np.mean(specificity_scores)
average_sensitivity = np.mean(sensitivity_scores)
average_precision = np.mean(precision_scores)
average_f1 = np.mean(f1_scores)

print("\n============== Final Averages ==============")
print(f"Average Accuracy: {average_accuracy:.6f}")
print(f"Average Specificity: {average_specificity:.6f}")
print(f"Average Sensitivity: {average_sensitivity:.6f}")
print(f"Average Precision: {average_precision:.6f}")
print(f"Average F1 Score: {average_f1:.6f}")


In [ ]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.applications import ResNet152
from tensorflow.keras.layers import GlobalMaxPooling2D, GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.models import Model

# --- ✅ EuPea Attention Module ---
class EuPea(tf.keras.layers.Layer):
    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):
        x_mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        x_std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)

        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)
        d_min = tf.reduce_min(d, axis=[1, 2], keepdims=True)
        d_max = tf.reduce_max(d, axis=[1, 2], keepdims=True)
        d_norm = (d - d_min) / (d_max - d_min + self.epsilon)

        z = tf.square((x - x_mean) / (x_std + self.epsilon))
        z_min = tf.reduce_min(z, axis=[1, 2], keepdims=True)
        z_max = tf.reduce_max(z, axis=[1, 2], keepdims=True)
        z_norm = (z - z_min) / (z_max - z_min + self.epsilon)

        attention = tf.nn.sigmoid(d_norm + z_norm)
        return x * attention

# --- ✅ Build ResNet152 Feature Extraction Model with EuPea ---
def build_eupea_resnet152_feature_model():
    base_model = ResNet152(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

    input_tensor = Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet.preprocess_input(input_tensor)

    backbone = Model(
        inputs=base_model.input,
        outputs=[
            base_model.get_layer("pool1_pool").output,          # 64 channels
            base_model.get_layer("conv2_block3_out").output,    # 256 channels
            base_model.get_layer("conv3_block8_out").output,    # 512 channels
            base_model.get_layer("conv4_block36_out").output,   # 1024 channels
            base_model.get_layer("conv5_block3_out").output     # 2048 channels
        ]
    )

    features = backbone(x)
    attention = EuPea()

    pooling_layers = [
        GlobalMaxPooling2D(),
        GlobalMaxPooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D(),
        GlobalAveragePooling2D()
    ]

    pooled_outputs = []
    for feat, pool in zip(features, pooling_layers):
        attn_feat = attention(feat)
        pooled_outputs.append(pool(attn_feat))

    final_feature = Concatenate()(pooled_outputs)
    return Model(inputs=input_tensor, outputs=final_feature)

# --- ✅ Instantiate Feature Model ---
eupea_feature_model = build_eupea_resnet152_feature_model()
print(eupea_feature_model.summary())

# --- ✅ Image Directory and Labels ---
# --- ✅ Define image directory and labels ---
image_dir = r"/kaggle/input/cervical-type/After Augmentation"
labels = ['Type - 1', 'Type - 2', 'Type - 3']
res_152_df = pd.DataFrame(columns=['features', 'labels'])

# --- ✅ Feature Extraction Function ---
def extract_features(image_path, label):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = tf.expand_dims(x, axis=0)

    features = eupea_feature_model(x)
    features_np = features.numpy().flatten()
    res_152_df.loc[len(res_152_df)] = [features_np, label]

# --- ✅ Process Dataset ---
for label_idx, label in enumerate(labels):
    label_dir = os.path.join(image_dir, label)
    for subdir, dirs, files in os.walk(label_dir):
        for file in files:
            extract_features(os.path.join(subdir, file), label_idx)

# --- ✅ Show Results ---
print(res_152_df.head())
print("Unique feature lengths:", res_152_df['features'].apply(len).unique())


In [ ]:
print("BF-AD-AP-Resnet152 dataframe", res_152_df)

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, f1_score

# Separate features and labels
# # Separate features and labels
X = res_152_df['features'].tolist()
y = res_152_df['labels'].tolist()


#print("ResNet152 with 10-fold cross-validation and linear SVM")

num_splits = 5

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

kfold = KFold(n_splits=num_splits, shuffle=True)
svm = SVC(kernel='linear')

for train_index, test_index in kfold.split(X):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    conf_matrix = confusion_matrix(y_test, y_pred)

    tp = conf_matrix[0, 0]
    fp = conf_matrix[1, 0] + conf_matrix[2, 0]
    fn = conf_matrix[0, 1] + conf_matrix[0, 2]
    tn = conf_matrix[1, 1] + conf_matrix[2, 2] + conf_matrix[1, 2] + conf_matrix[2, 1]

    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)

average_accuracy = np.mean(accuracy_scores)
average_specificity = np.mean(specificity_scores)
average_sensitivity = np.mean(sensitivity_scores)
average_precision = np.mean(precision_scores)
average_f1 = np.mean(f1_scores)

print(f"Average Precision: {average_precision:.6f}")
print(f"Average F1 Score: {average_f1:.6f}")
print(f"Average Specificity: {average_specificity:.6f}")
print(f"Average Sensitivity: {average_sensitivity:.6f}")
print(f"Average Accuracy: {average_accuracy:.6f}")


# Feature Fusion

In [ ]:
merged_features_list = []
min_length = min(len(res_50_df), len(res_152_df),len(res_101_df))
for i in range(min_length):
    merged_features = np.concatenate([res_50_df.loc[i, 'features'], res_152_df.loc[i, 'features'], res_101_df.loc[i, 'features']])
    merged_features_list.append(merged_features)

# Create a new dataframe with merged features and labels
merged_labels_list = res_50_df['labels'][:min_length]  # Using 'label' column from dct_df
merged_data = {'features': merged_features_list, 'labels': merged_labels_list}
merged_df= pd.DataFrame(merged_data)




In [ ]:
print("Length of features in merged_df:", len(merged_df['features'].iloc[-1]))

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix, precision_score, f1_score
from sklearn.preprocessing import MinMaxScaler

X = merged_df['features'].tolist()
y = merged_df['labels'].tolist()

#print("merged with 5-fold cross-validation and linear SVM normalization")

num_splits = 5


accuracy_scores = []
specificity_scores = []
sensitivity_scores = []
precision_scores = []
f1_scores = []

kfold = KFold(n_splits=num_splits, shuffle=True)
svm = SVC(kernel='linear')

for train_index, test_index in kfold.split(X):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]

    # Normalize the training data
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    X_test_scaled = scaler.transform(X_test)

    # Train the SVM model on normalized training data
    svm.fit(X_train_scaled, y_train)

    # Make predictions on the test data
    y_pred = svm.predict(X_test_scaled)

    conf_matrix = confusion_matrix(y_test, y_pred)

    tp = conf_matrix[0, 0]
    fp = conf_matrix[1, 0] + conf_matrix[2, 0]
    fn = conf_matrix[0, 1] + conf_matrix[0, 2]
    tn = conf_matrix[1, 1] + conf_matrix[2, 2] + conf_matrix[1, 2] + conf_matrix[2, 1]

    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    # Macro-averaged precision and F1-score
    precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)

average_accuracy = np.mean(accuracy_scores)
average_specificity = np.mean(specificity_scores)
average_sensitivity = np.mean(sensitivity_scores)
average_precision = np.mean(precision_scores)
average_f1 = np.mean(f1_scores)

print(f"Average Precision  : {average_precision:.6f}")
print(f"Average F1-Score   : {average_f1:.6f}")
print(f"Average Specificity: {average_specificity:.6f}")
print(f"Average Sensitivity: {average_sensitivity:.6f}")
print(f"\nAverage Accuracy   : {average_accuracy:.6f}")
